In [205]:
# import packages
import os
import pandas as pd
import numpy as np
import s3fs
import sklearn
import json
import ast
sklearn.__version__

'1.7.1'

In [206]:
# recuperation des donnees dans l'espace S3
os.environ["AWS_ACCESS_KEY_ID"] = '6DAEWSLI8LVXPW44KDA3'
os.environ["AWS_SECRET_ACCESS_KEY"] = 'l4gI75HRy+eWEC4SarJSoZa1b8PTdSwr9H3+s2j8'
os.environ["AWS_SESSION_TOKEN"] = 'eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJhY2Nlc3NLZXkiOiI2REFFV1NMSThMVlhQVzQ0S0RBMyIsImFjciI6IjAiLCJhbGxvd2VkLW9yaWdpbnMiOlsiKiJdLCJhdWQiOlsibWluaW8iLCJhY2NvdW50Il0sImF1dGhfdGltZSI6MTc1ODQ3MTYzNCwiYXpwIjoib255eGlhLW1pbmlvIiwiZW1haWwiOiJjcHJldm90QGFnaXJjLWFycmNvLmZyIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsImV4cCI6MTc1OTY4MjE0OCwiZmFtaWx5X25hbWUiOiJQUkVWT1QiLCJnaXZlbl9uYW1lIjoiQ8OpY2lsZSIsImlhdCI6MTc1ODQ3MjU0OCwiaXNzIjoiaHR0cHM6Ly9hdXRoLmdyb3VwZS1nZW5lcy5mci9yZWFsbXMvZ2VuZXMiLCJqdGkiOiIzODkzNjUyOS0wMGQzLTQ1M2UtODI1Ni00OWFiNDExYjU2N2YiLCJuYW1lIjoiQ8OpY2lsZSBQUkVWT1QiLCJwb2xpY3kiOiJzdHNvbmx5IiwicHJlZmVycmVkX3VzZXJuYW1lIjoiY3ByZXZvdC1lbnNhZSIsInJlYWxtX2FjY2VzcyI6eyJyb2xlcyI6WyJvZmZsaW5lX2FjY2VzcyIsImRlZmF1bHQtcm9sZXMtZ2VuZXMiLCJ1bWFfYXV0aG9yaXphdGlvbiJdfSwicmVzb3VyY2VfYWNjZXNzIjp7ImFjY291bnQiOnsicm9sZXMiOlsibWFuYWdlLWFjY291bnQiLCJtYW5hZ2UtYWNjb3VudC1saW5rcyIsInZpZXctcHJvZmlsZSJdfX0sInNjb3BlIjoib3BlbmlkIHByb2ZpbGUgZW1haWwiLCJzaWQiOiJmYzM0NmYyYi0wZWUzLTRkM2ItOTUzMi1mNGEyMjU5NGI5Y2MiLCJzdWIiOiIwZmUyODI0ZC0yOWFiLTQzZjQtODQ2Mi0zNWEwMDk0OWJiODAiLCJ0eXAiOiJCZWFyZXIifQ.MhJkwgn1hfqdDttmZbWuC6nPqmYNRj14nAEmukRov9OBhu1neZA38dBh3OjaMItZ-r73wEsiYY8LnIJ3NZw2Iw'
os.environ["AWS_DEFAULT_REGION"] = 'us-east-1'
fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio-simple.lab.groupe-genes.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

In [207]:
fs.ls('cprevot-ensae')
BUCKET = 'cprevot-ensae'
# recuperation de la table principale : movies_metadata 
FILE_KEY_S3 = '/projet/data/movies_metadata.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_movies = pd.read_csv(file_in,sep=',', header=0)

/tmp/ipykernel_6010/2993818107.py:7: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_movies = pd.read_csv(file_in,sep=',', header=0)


In [208]:
# filtre video=False status="released"
df = data_movies[(data_movies["video"] == False) & (data_movies["status"] == "Released")]

# suppression des colonnes non pertinentes
# revenue est considéré comme non pertinent car dans une optique de prédiction , cette information ne sera pas connue
df.drop(
    columns=['adult', 'homepage', 'overview', 'popularity','poster_path', 
            'revenue','spoken_languages','status', 'tagline',  'video' 
    ],
    inplace=True
)
df.shape

/tmp/ipykernel_6010/257143493.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(


(44921, 14)

In [209]:
# conversion de "budget" en nombre
df["budget"] = pd.to_numeric(df["budget"], errors="coerce")
print (df["budget"])

0        30000000
1        65000000
2               0
3        16000000
4               0
           ...   
45461           0
45462           0
45463           0
45464           0
45465           0
Name: budget, Length: 44921, dtype: int64


/tmp/ipykernel_6010/2137332106.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["budget"] = pd.to_numeric(df["budget"], errors="coerce")


In [210]:
# budget et duree (runtime) nuls a considerer comme valeurs manquantes
variables = ["budget", "runtime"]

for var in variables:
    nb_nan = df[var].isna().sum()
    nb_zero = (df[var] == 0).sum()
    print(f"{var} ➜ {nb_nan} valeurs manquantes, {nb_zero} valeurs égales à 0")

budget ➜ 0 valeurs manquantes, 36061 valeurs égales à 0
runtime ➜ 250 valeurs manquantes, 1486 valeurs égales à 0


In [211]:
df["budget"] = df["budget"].replace(0, np.nan)
df["runtime"] = df["runtime"].replace(0, np.nan)

/tmp/ipykernel_6010/1597220208.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["budget"] = df["budget"].replace(0, np.nan)
/tmp/ipykernel_6010/1597220208.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["runtime"] = df["runtime"].replace(0, np.nan)


#### traitement de la variable "belongs_to_collection"

In [212]:
# belongs to collection : 
df["belongs_to_collection"]

0        {'id': 10194, 'name': 'Toy Story Collection', ...
1                                                      NaN
2        {'id': 119050, 'name': 'Grumpy Old Men Collect...
3                                                      NaN
4        {'id': 96871, 'name': 'Father of the Bride Col...
                               ...                        
45461                                                  NaN
45462                                                  NaN
45463                                                  NaN
45464                                                  NaN
45465                                                  NaN
Name: belongs_to_collection, Length: 44921, dtype: object

In [213]:
# belongs to collection : 
# on crée un booleen indic_collec 
df["indic_collec"]  = df["belongs_to_collection"].notna().astype("int8") # int8 → 1 octet par valeur



/tmp/ipykernel_6010/3848633535.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["indic_collec"]  = df["belongs_to_collection"].notna().astype("int8") # int8 → 1 octet par valeur


In [214]:
df["indic_collec"] 

0        1
1        0
2        1
3        0
4        1
        ..
45461    0
45462    0
45463    0
45464    0
45465    0
Name: indic_collec, Length: 44921, dtype: int8

In [215]:
df["indic_collec"].value_counts()

indic_collec
0    40459
1     4462
Name: count, dtype: int64

In [216]:
# sous-df avec les valeurs non vides de belongs_to_collection
df_collec = df[df["indic_collec"] == 1]

In [217]:
df_collec['belongs_to_collection'] = df_collec['belongs_to_collection'].apply(ast.literal_eval)

/tmp/ipykernel_6010/1639396710.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_collec['belongs_to_collection'] = df_collec['belongs_to_collection'].apply(ast.literal_eval)


In [218]:
df['belongs_to_collection']

0        {'id': 10194, 'name': 'Toy Story Collection', ...
1                                                      NaN
2        {'id': 119050, 'name': 'Grumpy Old Men Collect...
3                                                      NaN
4        {'id': 96871, 'name': 'Father of the Bride Col...
                               ...                        
45461                                                  NaN
45462                                                  NaN
45463                                                  NaN
45464                                                  NaN
45465                                                  NaN
Name: belongs_to_collection, Length: 44921, dtype: object

In [219]:
# on cree une fonction pour recuperer le nom de la collection s il est rempli, rien sinon
def recup_collec(x):
    if pd.notna(x):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return None
    return x

df['belongs_to_collection'] = df['belongs_to_collection'].apply(recup_collec)

/tmp/ipykernel_6010/2237938526.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['belongs_to_collection'] = df['belongs_to_collection'].apply(recup_collec)


In [220]:
df['nom_collec'] = df['belongs_to_collection'].apply(
            lambda x: x['name'] if isinstance(x, dict) and 'name' in x else None
                                                )

/tmp/ipykernel_6010/2993967180.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['nom_collec'] = df['belongs_to_collection'].apply(


In [221]:
df['nom_collec']

0                  Toy Story Collection
1                                  None
2             Grumpy Old Men Collection
3                                  None
4        Father of the Bride Collection
                      ...              
45461                              None
45462                              None
45463                              None
45464                              None
45465                              None
Name: nom_collec, Length: 44921, dtype: object

In [222]:
# creation d'une colonne rang
# on trie par collec et date de sortie
df = df.sort_values(by=['nom_collec', 'release_date'], ascending=[True, True])
df["rang"] = np.nan # initialisation
masque = df["nom_collec"].notna() # si collec n'est pas manquant
df.loc[masque, "rang"] = df.loc[masque].groupby("nom_collec").cumcount() + 1
df.rang.value_counts(dropna=False)

rang
NaN     40459
1.0      1690
2.0      1298
3.0       618
4.0       275
5.0       162
6.0       101
7.0        63
8.0        45
9.0        35
10.0       26
11.0       19
12.0       19
13.0       15
14.0       13
15.0       11
16.0        8
17.0        7
19.0        7
20.0        7
21.0        7
18.0        7
22.0        6
23.0        5
25.0        5
24.0        5
26.0        4
27.0        2
28.0        1
29.0        1
Name: count, dtype: int64

In [223]:
df.columns

Index(['belongs_to_collection', 'budget', 'genres', 'id', 'imdb_id',
       'original_language', 'original_title', 'production_companies',
       'production_countries', 'release_date', 'runtime', 'title',
       'vote_average', 'vote_count', 'indic_collec', 'nom_collec', 'rang'],
      dtype='object')

In [224]:
# on rajoute une colonne rang max
df_max_rang = df.groupby('nom_collec')['rang'].max().reset_index()
df_max_rang.rename(columns={'rang': 'max_rang'}, inplace=True)
df = df.merge(df_max_rang, on='nom_collec', how='left')

In [225]:
#  ceux qui n'ont qu'un seul opus present sont a considerer comme ne faisant pas partie d'une serie
condition = (df['rang'] == 1) & (df['max_rang'] == 1)

df.loc[condition, 'indic_collec'] = 0
df.loc[condition, 'nom_collec'] = None
df.loc[condition, 'rang'] = np.nan
df.loc[condition, 'max_rang'] = np.nan

df['max_rang']


0        2.0
1        2.0
2        NaN
3        NaN
4        2.0
        ... 
44916    NaN
44917    NaN
44918    NaN
44919    NaN
44920    NaN
Name: max_rang, Length: 44921, dtype: float64

In [226]:
print(f"Lignes modifiées : {condition.sum()}")

Lignes modifiées : 392


In [227]:
# pour les sagas, on recupere la note de l opus precedent
df_prec = df[df['rang'] >= 1]

df_prec['rang'] += 1  # rang dans df_prec correspondra à rang - 1 dans df
df_prec = df_prec.rename(columns={'vote_average': 'vote_prec'}) # colonne vote_average renommée poru etre gardee lors de la fusion
df_prec = df_prec[['rang','nom_collec','vote_prec']]

/tmp/ipykernel_6010/183179955.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prec['rang'] += 1  # rang dans df_prec correspondra à rang - 1 dans df


In [228]:
df_prec

,rang,nom_collec,vote_prec
0,2.0,... Has Fallen Collection,6.2
1,3.0,... Has Fallen Collection,5.8
4,2.0,100 Girls Collection,5.6
5,3.0,100 Girls Collection,4.7
6,2.0,101 Dalmatians (Animated) Collection,6.8
...,...,...,...
4457,2.0,男はつらいよ シリーズ,7.0
4458,3.0,男はつらいよ シリーズ,0.0
4459,4.0,男はつらいよ シリーズ,0.0
4460,2.0,식객 시리즈,5.0


In [229]:

# Merge sur nom_collec et rang
df2 = df.merge(df_prec[['nom_collec', 'rang', 'vote_prec']],
                     on=['nom_collec', 'rang'],
                     how='left')

In [230]:
df = df2.drop(columns=['belongs_to_collection'])

In [231]:
df.columns

Index(['budget', 'genres', 'id', 'imdb_id', 'original_language',
       'original_title', 'production_companies', 'production_countries',
       'release_date', 'runtime', 'title', 'vote_average', 'vote_count',
       'indic_collec', 'nom_collec', 'rang', 'max_rang', 'vote_prec'],
      dtype='object')

#### traitement de la variable "genres"

In [232]:
# recuperation des infos genre, maison de prod et pays dans des dictionnaires
# y a t il des variables manquantes ? 
variables = ["genres", "production_companies","production_countries"]

for var in variables:
    nb_nan = df[var].isna().sum()
    print(f"{var} ➜ {nb_nan} valeurs manquantes")


genres ➜ 0 valeurs manquantes
production_companies ➜ 0 valeurs manquantes
production_countries ➜ 0 valeurs manquantes


In [233]:
# variable genres : a considerer comme une liste de dictionnaire
df['genres'] = df['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
# : il peut y en avoir plusieurs : combien maximum?
max_genres = df['genres'].apply(lambda x: len(x) if isinstance(x, list) else 0).max()
print(f"Nombre maximum de dictionnaires dans 'genres' : {max_genres}")


Nombre maximum de dictionnaires dans 'genres' : 8


In [234]:
# Creation des colonnes genre1 à genre8
for i in range(8):
    col_name = f'genre{i+1}'
    df[col_name] = df['genres'].apply(
        lambda x: x[i] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

In [236]:
df['genres'] = df['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
df['genre1'] = df['genres'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 and isinstance(x[0], dict) else None
)

In [237]:
  nb_nan = df["genre1"].isna().sum()
  print(f"{nb_nan} valeurs manquantes")
 

2369 valeurs manquantes


In [238]:
# comment se fait il qu'on a des valeurs manquantes ? 
df_manq = df[df['genre1'].isna()]
df_manq.shape


(2369, 26)

In [239]:
df_manq.head()
#pour voir un exemple on regarde id=81246 dans la table d'origine
exemple = data_movies[(data_movies["id"] == '81246')]
exemple2= df[(df["id"] == '81246') ]
exemple2


,budget,genres,id,imdb_id,original_language,original_title,production_companies,production_countries,release_date,runtime,...,max_rang,vote_prec,genre1,genre2,genre3,genre4,genre5,genre6,genre7,genre8
121,NaN,[],81246,tt0118582,en,Alien Nation: The Udara Legacy,[],[],1997-07-29,90.0,...,6.0,5.7,None,None,None,None,None,None,None,None


In [254]:
for i in range(1, 9):
    col = f'genre{i}'
    df[col] = df[col].apply(
        lambda x: x['name'] if isinstance(x, dict) and 'name' in x else None
    )


In [256]:
df['genre8']

0        None
1        None
2        None
3        None
4        None
         ... 
44916    None
44917    None
44918    None
44919    None
44920    None
Name: genre8, Length: 44921, dtype: object

#### traitement de la variable "production_companies"

In [240]:
# variable production_companies : a considerer comme une liste de dictionnaire
df['production_companies'] = df['production_companies'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
# : il peut y en avoir plusieurs : combien maximum?
max_prod = df['production_companies'].apply(lambda x: len(x) if isinstance(x, list) else 0).max()
print(f"Nombre maximum de dictionnaires dans 'production_companies' : {max_prod}")
# a voir : il ne sera pas pertinent de garder 26 colonnes

Nombre maximum de dictionnaires dans 'production_companies' : 26


In [241]:
# Creation des colonnes prod1 à prod26
for i in range(26):
    col_name = f'prod{i+1}'
    df[col_name] = df['production_companies'].apply(
        lambda x: x[i] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

In [242]:
nb_nan = df["prod1"].isna().sum()
print(f"{nb_nan} valeurs manquantes")

11580 valeurs manquantes


In [248]:
for i in range(1, 27):
    col = f'prod{i}'
    if col in df.columns:
        nb_nan = df[col].isna().sum()
        print(f"{col} : {nb_nan} valeurs manquantes")
    else:
        print(f"{col} : colonne absente du DataFrame")

prod1 : 11580 valeurs manquantes
prod2 : 28029 valeurs manquantes
prod3 : 35936 valeurs manquantes
prod4 : 40619 valeurs manquantes
prod5 : 42553 valeurs manquantes
prod6 : 43499 valeurs manquantes
prod7 : 44060 valeurs manquantes
prod8 : 44370 valeurs manquantes
prod9 : 44548 valeurs manquantes
prod10 : 44681 valeurs manquantes
prod11 : 44748 valeurs manquantes
prod12 : 44790 valeurs manquantes
prod13 : 44825 valeurs manquantes
prod14 : 44844 valeurs manquantes
prod15 : 44860 valeurs manquantes
prod16 : 44869 valeurs manquantes
prod17 : 44889 valeurs manquantes
prod18 : 44896 valeurs manquantes
prod19 : 44899 valeurs manquantes
prod20 : 44904 valeurs manquantes
prod21 : 44909 valeurs manquantes
prod22 : 44913 valeurs manquantes
prod23 : 44916 valeurs manquantes
prod24 : 44916 valeurs manquantes
prod25 : 44917 valeurs manquantes
prod26 : 44918 valeurs manquantes


In [258]:
for i in range(1, 27):
    col = f'prod{i}'
    df[col] = df[col].apply(
        lambda x: x['name'] if isinstance(x, dict) and 'name' in x else None
    )
df.prod26

0        None
1        None
2        None
3        None
4        None
         ... 
44916    None
44917    None
44918    None
44919    None
44920    None
Name: prod26, Length: 44921, dtype: object

#### traitement de la variable "production_countries"

In [244]:
# variable production_countries : a considerer comme une liste de dictionnaire
df['production_countries'] = df['production_countries'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
# : il peut y en avoir plusieurs : combien maximum?
max_prod = df['production_countries'].apply(lambda x: len(x) if isinstance(x, list) else 0).max()
print(f"Nombre maximum de dictionnaires dans 'production_countries' : {max_prod}")
# a voir : il ne sera pas pertinent de garder 25 colonnes

Nombre maximum de dictionnaires dans 'production_countries' : 25


In [245]:
# Creation des colonnes pays1 a pays25
for i in range(25):
    col_name = f'pays{i+1}'
    df[col_name] = df['production_countries'].apply(
        lambda x: x[i] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

In [246]:
nb_nan = df["pays1"].isna().sum()
print(f"pays 1 : {nb_nan} valeurs manquantes")

pays 1 : 6110 valeurs manquantes


In [247]:
for i in range(1, 26):
    col = f'pays{i}'
    if col in df.columns:
        nb_nan = df[col].isna().sum()
        print(f"{col} : {nb_nan} valeurs manquantes")
    else:
        print(f"{col} : colonne absente du DataFrame")


pays1 : 6110 valeurs manquantes
pays2 : 37944 valeurs manquantes
pays3 : 42783 valeurs manquantes
pays4 : 44239 valeurs manquantes
pays5 : 44701 valeurs manquantes
pays6 : 44852 valeurs manquantes
pays7 : 44891 valeurs manquantes
pays8 : 44903 valeurs manquantes
pays9 : 44909 valeurs manquantes
pays10 : 44915 valeurs manquantes
pays11 : 44916 valeurs manquantes
pays12 : 44918 valeurs manquantes
pays13 : 44919 valeurs manquantes
pays14 : 44919 valeurs manquantes
pays15 : 44919 valeurs manquantes
pays16 : 44920 valeurs manquantes
pays17 : 44920 valeurs manquantes
pays18 : 44920 valeurs manquantes
pays19 : 44920 valeurs manquantes
pays20 : 44920 valeurs manquantes
pays21 : 44920 valeurs manquantes
pays22 : 44920 valeurs manquantes
pays23 : 44920 valeurs manquantes
pays24 : 44920 valeurs manquantes
pays25 : 44920 valeurs manquantes


In [249]:
df.shape

(44921, 77)

In [259]:
for i in range(1, 26):
    col = f'pays{i}'
    df[col] = df[col].apply(
        lambda x: x['name'] if isinstance(x, dict) and 'name' in x else None
    )
df.pays25

0        None
1        None
2        None
3        None
4        None
         ... 
44916    None
44917    None
44918    None
44919    None
44920    None
Name: pays25, Length: 44921, dtype: object

In [261]:
valeurs_uniques = df['pays1'].dropna().unique().tolist()
print(valeurs_uniques)
len(valeurs_uniques)

['United States of America', 'Bulgaria', 'Germany', 'United Kingdom', 'India', 'France', 'Japan', 'Spain', 'New Zealand', 'Hong Kong', 'Canada', 'Australia', 'Indonesia', 'Russia', 'Finland', 'Italy', 'Ireland', 'Netherlands', 'Denmark', 'Brazil', 'Sweden', 'Hungary', 'Thailand', 'Belgium', 'Argentina', 'Mexico', 'Austria', 'Greece', 'South Africa', 'United Arab Emirates', 'Taiwan', 'Norway', 'Romania', 'China', 'South Korea', 'Czech Republic', 'Luxembourg', 'Turkey', 'Morocco', 'Switzerland', 'Poland', 'Bahamas', 'Iran', 'Israel', 'Bangladesh', 'Senegal', 'Serbia', 'Estonia', 'Philippines', 'Peru', 'Portugal', 'Botswana', 'Ukraine', 'Georgia', 'Soviet Union', 'East Germany', 'Egypt', 'Singapore', 'Cuba', 'Venezuela', 'Croatia', 'Trinidad and Tobago', 'Slovenia', 'Bosnia and Herzegovina', 'Kyrgyz Republic', 'Monaco', 'Lithuania', 'Yugoslavia', 'Uzbekistan', 'Bolivia', 'Mauritania', 'Pakistan', 'Czechoslovakia', 'Algeria', 'Slovakia', 'Armenia', 'Liechtenstein', 'Latvia', 'Jamaica', 'Ca

143